# SWOT WSE Error Correction in Urban Areas Using QRF

PI: Kostas Andreadis, UMass Amherst  
Student: Kwanhyuck Kim  
HPC: Unity cluster, UMass Amherst

---

Pipeline Overview:
1. Phase 0: Setup & Configuration
2. Phase 1: Study Area Definition (SWORD -> Paired Reaches -> Buffer -> FctImp -> Study Reaches)
3. Phase 2: SWOT Reach & Node Data
4. Phase 3: USGS Gage Height (Time-Targeted)
5. Phase 4: Design Filter + Random Train/Test Split
6. Phase 5: Feature Engineering (~33 features, quality flags as features)
7. Phase 6: QRF Training + k-Fold Cross-Validation
8. Phase 7: WSE Correction & Physics Recalculation
9. Phase 8: Validation (overall + FctImp-stratified + PI calibration)

---
## Phase 0: Setup & Configuration

In [ ]:
# Phase 0: All imports
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import glob
import time
import gc
import warnings
import os
import subprocess
import requests
from io import StringIO
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

from shapely.geometry import LineString, Polygon
from shapely.ops import unary_union
from joblib import Parallel, delayed
from exactextract import exact_extract
import rasterio
import pyarrow.parquet as pq
import pyarrow as pa
import dataretrieval.nwis as nwis

from IPython.display import clear_output, display

warnings.filterwarnings('ignore')

print("All imports successful.")

In [ ]:
# Phase 0: Configuration  (*** EDIT THIS CELL ONLY ***)

# --- Paired CSV (reach_id <-> gage_id mapping) ---
PAIRED_CSV = "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/5_sword_usgs/paired_reach_SWOT_gage.csv"

# --- SWORD shapefiles ---
SWORD_PATH = "/nas/cee-ice/data/SWORD/SWORD_v17b/shp"

# --- NLCD FctImp raster ---
NLCD_FCTIMP = "/work/pi_kandread_umass_edu/swot-urban/0_data_collection/1_nlcd/Annual_NLCD_FctImp_2024_CU_C1V1.tif"

# --- SWOT pre-downloaded data ---
SWOT_REACH_FILE = "/nas/cee-hydro/floods/vD/SP_reaches_vD_all.parquet"
SWOT_NODE_FILE  = "/nas/cee-hydro/floods/vD/SP_nodes_vD_NA.parquet"

# --- SWOT download scripts (if data not yet downloaded) ---
REACH_DOWNLOAD_SCRIPT = Path("/Users/targetkim/target/wse-correction/swot_hydrocron_download_reach.py")
NODE_DOWNLOAD_SCRIPT  = Path("/Users/targetkim/target/wse-correction/swot_hydrocron_download_node.py")

# --- USGS matched (pre-existing, optional) ---
USGS_MATCHED_FILE = "/nas/cee-hydro/floods/vD/usgs_gage_height_matched_vD.parquet"

# --- Output ---
OUTPUT_DIR = Path("/nas/cee-hydro/floods/vD")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- Hydrocron API ---
HYDROCRON_URL = "https://soto.podaac.earthdatacloud.nasa.gov/hydrocron/v1/timeseries"
COLLECTION_NAME = "SWOT_L2_HR_RiverSP_D"
START_TIME = "2023-07-27T00:00:00Z"
END_TIME   = "2026-02-14T23:59:59Z"

# --- Parameters ---
BUFFER_MULTIPLIER = 2.5        # corridor buffer = width * BUFFER_MULTIPLIER each side
TEMPORAL_TOLERANCE = "3h"      # merge_asof tolerance for SWOT-USGS matching
HAMPEL_H = 3                   # Hampel half-window (observations)
HAMPEL_K = 7                   # Hampel threshold (MAD units)
HAMPEL_GAP_DAYS = 3            # Max gap per observation in Hampel window

print("Configuration loaded.")

---
## Phase 1: Study Area Definition

SWORD -> Paired Reaches -> Corridor Buffer -> FctImp -> Study Reaches

### 1.1 Load Paired CSV and Filter SWORD

In [ ]:
# ---- Load paired CSV ----
paired = pd.read_csv(PAIRED_CSV, dtype={'gage_id': object}, low_memory=False)
print(f"Paired CSV loaded: {len(paired):,} rows, {paired.columns.tolist()}")

paired_reach_ids = paired['reach_id'].drop_duplicates().tolist()
paired_gage_ids  = paired[paired.gage_agency == 'USGS']['gage_id'].drop_duplicates().tolist()
id_link = paired[['reach_id', 'gage_id']].drop_duplicates()

print(f"Unique paired reach_ids : {len(paired_reach_ids):,}")
print(f"Unique USGS gage_ids   : {len(paired_gage_ids):,}")
print(f"id_link rows           : {len(id_link):,}")

In [ ]:
# ---- Load SWORD NA reaches (glob *reaches*.shp) ----
reaches_pattern = str(Path(SWORD_PATH) / 'NA' / '*reaches*.shp')
reaches_files = sorted(glob.glob(reaches_pattern))

if not reaches_files:
    raise FileNotFoundError(f"No reaches shapefiles found in {SWORD_PATH}/NA/")

print(f"Found {len(reaches_files)} SWORD NA reaches shapefiles")

reaches_list = [gpd.read_file(f) for f in reaches_files]
sword = gpd.GeoDataFrame(
    pd.concat(reaches_list, ignore_index=True),
    crs=reaches_list[0].crs
)
del reaches_list
gc.collect()

n_total_sword = len(sword)
print(f"Total SWORD NA reaches: {n_total_sword:,}")
print(f"CRS: {sword.crs}")

# ---- Filter to paired reach_ids only ----
sword = sword[sword['reach_id'].isin(paired_reach_ids)].copy()
n_paired = len(sword)
n_removed = n_total_sword - n_paired
pct_removed = n_removed / n_total_sword * 100

print(f"\nFilter to paired reaches: {n_total_sword:,} -> {n_paired:,} reaches "
      f"(removed {n_removed:,}, {pct_removed:.1f}%)")

### 1.2 Clean Invalid Widths

In [ ]:
n_before = len(sword)

# ---- Remove width in {0, -1} ----
invalid_zero_mask = sword['width'].isin([0, -1])
n_zero = invalid_zero_mask.sum()
sword = sword[~invalid_zero_mask].copy()
print(f"[1] Removed width == 0 or -1: {n_zero:,} reaches")

# ---- Remove width > max_width ----
invalid_max_mask = sword['width'] > sword['max_width']
n_max = invalid_max_mask.sum()
sword = sword[~invalid_max_mask].copy()
print(f"[2] Removed width > max_width: {n_max:,} reaches")

n_after = len(sword)
n_total_removed = n_before - n_after
pct = n_total_removed / n_before * 100 if n_before > 0 else 0
print(f"\nWidth cleanup: {n_before:,} -> {n_after:,} reaches "
      f"(removed {n_total_removed:,}, {pct:.1f}%)")

print(f"\nWidth statistics after cleanup:")
print(sword['width'].describe())

### 1.3 Corridor Buffer (width x 2.5 each side)

In [ ]:
def create_corner_arc(p_prev, p_curr, p_next, perp_direction, half_width, num_points=40):
    """Create circular arc polygon at corner (turning point)."""
    v1 = p_curr - p_prev
    v2 = p_next - p_curr
    v1_len = np.linalg.norm(v1)
    v2_len = np.linalg.norm(v2)
    if v1_len < 1e-10 or v2_len < 1e-10:
        return None
    v1_unit = v1 / v1_len
    v2_unit = v2 / v2_len
    perp1_base = np.array([-v1_unit[1], v1_unit[0]])
    perp2_base = np.array([-v2_unit[1], v2_unit[0]])
    if perp_direction == "right":
        perp1_base = -perp1_base
        perp2_base = -perp2_base
    arc_points = []
    for t in np.linspace(0, 1, num_points):
        perp_interp = (1 - t) * perp1_base + t * perp2_base
        perp_norm = np.linalg.norm(perp_interp)
        if perp_norm > 1e-10:
            arc_points.append(p_curr + (perp_interp / perp_norm) * half_width)
    arc_points.append(p_curr)
    if len(arc_points) < 3:
        return None
    try:
        arc = Polygon(arc_points)
        if not arc.is_valid:
            arc = arc.buffer(0)
        if arc.is_valid and arc.area > 1e-6:
            return arc
    except:
        pass
    return None


def process_reach(reach_id, linestring, width, buffer_multiplier=2.5):
    """Corridor buffer for single reach."""
    try:
        if linestring.is_empty or linestring.length == 0:
            return (reach_id, None)
        half_width = width * buffer_multiplier
        coords = np.array(linestring.coords, dtype=float)
        segment_buffers = [
            LineString([coords[i], coords[i+1]]).buffer(half_width, cap_style=2, join_style=1)
            for i in range(len(coords) - 1)
        ]
        corner_arcs = []
        for i in range(1, len(coords) - 1):
            for direction in ["left", "right"]:
                arc = create_corner_arc(coords[i-1], coords[i], coords[i+1], direction, half_width)
                if arc is not None:
                    corner_arcs.append(arc)
        segment_union = unary_union(segment_buffers)
        arc_union = unary_union(corner_arcs) if corner_arcs else None
        buffer_geom = unary_union([segment_union, arc_union]) if arc_union else segment_union
        if not buffer_geom.is_valid:
            buffer_geom = buffer_geom.buffer(0)
        return (reach_id, buffer_geom)
    except Exception as e:
        return (reach_id, None)


print("Buffer functions defined.")

In [ ]:
# ---- Project to EPSG:5070 for meter-based buffering ----
sword_proj = sword.copy()
if sword_proj.crs is None:
    sword_proj = sword_proj.set_crs("EPSG:4326")
sword_proj = sword_proj.to_crs("EPSG:5070")
crs_meters = sword_proj.crs

print(f"Processing {len(sword_proj):,} paired reaches with parallel buffering...")
print(f"Buffer multiplier: {BUFFER_MULTIPLIER} (half_width = width * {BUFFER_MULTIPLIER})")

start_time_buf = time.time()

results = Parallel(n_jobs=-1, backend="threading", verbose=5)(
    delayed(process_reach)(row['reach_id'], row.geometry, row['width'], BUFFER_MULTIPLIER)
    for _, row in sword_proj.iterrows()
)

elapsed = time.time() - start_time_buf

# ---- Unpack results ----
buffer_geoms = {r[0]: r[1] for r in results}
sword_proj['buffer_geometry'] = sword_proj['reach_id'].map(buffer_geoms)

success_count = sum(1 for v in buffer_geoms.values() if v is not None)
failed_count  = sum(1 for v in buffer_geoms.values() if v is None)

print(f"Buffer computation complete!")
print(f"  Successful: {success_count:,} reaches")
print(f"  Failed    : {failed_count:,} reaches")
print(f"  Time      : {elapsed:.1f} seconds (~{elapsed/60:.1f} min)")

# Drop reaches with failed buffer
n_before_buf = len(sword_proj)
sword_proj = sword_proj[sword_proj['buffer_geometry'].notna()].copy()
n_after_buf = len(sword_proj)
print(f"\nBuffer drop: {n_before_buf:,} -> {n_after_buf:,} reaches "
      f"(removed {n_before_buf - n_after_buf:,})")

# Set buffer_geometry as active geometry for extraction
sword_buffered = sword_proj.set_geometry('buffer_geometry')
sword_buffered.crs = crs_meters

### 1.4 FctImp Extraction (NLCD 2024)

In [ ]:
# ---- Check NLCD CRS and reproject buffers if needed ----
with rasterio.open(NLCD_FCTIMP) as src:
    nlcd_crs = src.crs
    print(f"NLCD FctImp CRS: {nlcd_crs}")

print(f"Buffer GDF CRS : {sword_buffered.crs}")

if sword_buffered.crs != nlcd_crs:
    print(f"CRS mismatch -> reprojecting buffers to {nlcd_crs} ...")
    sword_buffered = sword_buffered.to_crs(nlcd_crs)
    print(f"Reprojection complete. New CRS: {sword_buffered.crs}")
else:
    print("CRS already matches.")

# Fix any invalid geometries
invalid_mask = ~sword_buffered.geometry.is_valid
n_invalid = invalid_mask.sum()
if n_invalid > 0:
    print(f"Fixing {n_invalid} invalid geometries...")
    sword_buffered.loc[invalid_mask, 'buffer_geometry'] = (
        sword_buffered.loc[invalid_mask, 'buffer_geometry'].buffer(0)
    )
    print(f"Fixed. Remaining invalid: {(~sword_buffered.geometry.is_valid).sum()}")

# Add string ID for exact_extract
sword_buffered['reach_id_str'] = sword_buffered['reach_id'].astype(str)

In [ ]:
# ---- FctImp extraction in batches of 100 ----
batch_size = 100
n_total = len(sword_buffered)
n_batches = (n_total + batch_size - 1) // batch_size

print(f"Extracting FctImp for {n_total:,} reaches in {n_batches} batches...")

all_stats = []
t0 = time.time()

for i in range(n_batches):
    start_idx = i * batch_size
    end_idx = min((i + 1) * batch_size, n_total)
    gdf_batch = sword_buffered.iloc[start_idx:end_idx]
    
    stats = exact_extract(
        NLCD_FCTIMP,
        gdf_batch,
        ['mean'],
        include_cols=['reach_id_str'],
        output='pandas'
    )
    all_stats.append(stats)
    
    if (i + 1) % 50 == 0 or (i + 1) == n_batches:
        clear_output(wait=True)
        print(f"FctImp extraction: batch {i+1}/{n_batches} "
              f"({end_idx}/{n_total} reaches, {time.time()-t0:.0f}s)")

fctimp_stats = pd.concat(all_stats, ignore_index=True)
fctimp_stats = fctimp_stats.rename(columns={'mean': 'fctimp_mean'})

elapsed_ext = time.time() - t0
print(f"\nFctImp extraction complete: {len(fctimp_stats):,} reaches, {elapsed_ext:.0f}s")

# ---- Merge back to sword ----
sword['reach_id_str'] = sword['reach_id'].astype(str)
sword = sword.merge(fctimp_stats, on='reach_id_str', how='left')

print(f"\nFctImp statistics:")
print(sword['fctimp_mean'].describe())

### 1.5 Study Reach Selection

In [ ]:
n_paired_total = len(sword)

# Study reaches = those with FctImp > 0 (some urban footprint)
study_mask = sword['fctimp_mean'] > 0
study_reaches = sword[study_mask].copy()
non_urban = sword[~study_mask].copy()

n_study = len(study_reaches)
n_non_urban = len(non_urban)

print(f"Study reaches (FctImp > 0): {n_study:,} out of {n_paired_total:,} paired reaches "
      f"({n_study/n_paired_total*100:.1f}%)")
print(f"Non-urban reaches (FctImp = 0): {n_non_urban:,} reaches "
      f"({n_non_urban/n_paired_total*100:.1f}%)")

# Save study reach IDs for later
study_reach_ids = study_reaches['reach_id'].tolist()

print(f"\nStudy reach FctImp statistics:")
print(study_reaches['fctimp_mean'].describe())

# Clean up large objects
del sword_proj, sword_buffered
gc.collect()
print(f"\nMemory cleaned.")

---
## Phase 2: SWOT Reach & Node Data

### 2.1 Load or Download SWOT Reach Data

In [ ]:
if Path(SWOT_REACH_FILE).exists():
    print(f"Loading pre-downloaded SWOT reach data from:\n  {SWOT_REACH_FILE}")
    t0 = time.time()
    swot_reach = pd.read_parquet(SWOT_REACH_FILE)
    print(f"  Loaded in {time.time()-t0:.1f}s")
    
    n_before_filter = len(swot_reach)
    n_reaches_before = swot_reach['reach_id'].nunique()
    
    # Filter to study reach IDs
    swot_reach = swot_reach[swot_reach['reach_id'].isin(study_reach_ids)].copy()
    
    n_after_filter = len(swot_reach)
    n_reaches_after = swot_reach['reach_id'].nunique()
    n_removed = n_before_filter - n_after_filter
    
    print(f"\n  Filter to study reaches:")
    print(f"    Observations: {n_before_filter:,} -> {n_after_filter:,} "
          f"(removed {n_removed:,}, {n_removed/n_before_filter*100:.1f}%)")
    print(f"    Reaches     : {n_reaches_before:,} -> {n_reaches_after:,}")
    
    # Parse time
    if 'time_str' in swot_reach.columns:
        swot_reach['time_dt'] = pd.to_datetime(swot_reach['time_str'])
    
    print(f"\n  Shape     : {swot_reach.shape}")
    print(f"  Columns   : {len(swot_reach.columns)}")
    if 'time_dt' in swot_reach.columns:
        print(f"  Time range: {swot_reach['time_dt'].min()} to {swot_reach['time_dt'].max()}")
else:
    print(f"SWOT reach file not found: {SWOT_REACH_FILE}")
    print(f"Run download script: python {REACH_DOWNLOAD_SCRIPT}")
    print(f"Or use the in-notebook download in Section 2.2")

In [ ]:
# ---- Load SWOT Node data (optional, for slope recalculation) ----
if Path(SWOT_NODE_FILE).exists():
    print(f"Loading SWOT node data from:\n  {SWOT_NODE_FILE}")
    t0 = time.time()
    swot_node = pd.read_parquet(SWOT_NODE_FILE)
    print(f"  Loaded in {time.time()-t0:.1f}s")
    
    n_before_node = len(swot_node)
    swot_node = swot_node[swot_node['reach_id'].isin(study_reach_ids)].copy()
    n_after_node = len(swot_node)
    n_removed_node = n_before_node - n_after_node
    
    print(f"  Filter to study reaches: {n_before_node:,} -> {n_after_node:,} "
          f"(removed {n_removed_node:,}, {n_removed_node/n_before_node*100:.1f}%)")
    print(f"  Shape: {swot_node.shape}")
else:
    print(f"SWOT node file not found: {SWOT_NODE_FILE}")
    print(f"Run: python {NODE_DOWNLOAD_SCRIPT} NA")
    swot_node = None

### 2.2 Alternative: In-Notebook Download (if files not available)

In [ ]:
def query_hydrocron(feature_id, query_url, start_time, end_time, fields, 
                    collection_name, feature_type="Reach"):
    """Query Hydrocron API for a single feature's time series."""
    params = {
        "feature": feature_type,
        "feature_id": feature_id,
        "output": "csv",
        "start_time": start_time,
        "end_time": end_time,
        "fields": fields,
        "collection_name": collection_name
    }
    try:
        response = requests.get(query_url, params=params, timeout=60)
        response.raise_for_status()
        data = response.json()
        if "results" in data and data["results"].get("csv"):
            return pd.read_csv(StringIO(data["results"]["csv"]))
    except:
        pass
    return None


def download_swot_reaches_notebook(reach_ids, output_file, max_workers=100, chunk_size=2000):
    """Download SWOT reach data using ThreadPoolExecutor with chunk saves."""
    
    # SWOT reach fields
    all_fields = (
        "reach_id,time,time_tai,time_str,cycle_id,pass_id,"
        "p_lat,p_lon,river_name,"
        "wse,wse_u,wse_r_u,wse_c,wse_c_u,"
        "slope,slope_u,slope_r_u,slope2,slope2_u,slope2_r_u,"
        "width,width_u,width_c,width_c_u,"
        "area_total,area_tot_u,area_detct,area_det_u,area_wse,"
        "d_x_area,d_x_area_u,"
        "layovr_val,node_dist,loc_offset,xtrk_dist,"
        "dschg_c,dschg_c_u,dschg_csf,dschg_c_q,"
        "dschg_gc,dschg_gc_u,dschg_gcsf,dschg_gc_q,"
        "dschg_m,dschg_m_u,dschg_msf,dschg_m_q,"
        "dschg_gm,dschg_gm_u,dschg_gmsf,dschg_gm_q,"
        "dschg_b,dschg_b_u,dschg_bsf,dschg_b_q,"
        "dschg_gb,dschg_gb_u,dschg_gbsf,dschg_gb_q,"
        "dschg_h,dschg_h_u,dschg_hsf,dschg_h_q,"
        "dschg_gh,dschg_gh_u,dschg_ghsf,dschg_gh_q,"
        "dschg_o,dschg_o_u,dschg_osf,dschg_o_q,"
        "dschg_go,dschg_go_u,dschg_gosf,dschg_go_q,"
        "dschg_s,dschg_s_u,dschg_ssf,dschg_s_q,"
        "dschg_gs,dschg_gs_u,dschg_gssf,dschg_gs_q,"
        "dschg_i,dschg_i_u,dschg_isf,dschg_i_q,"
        "dschg_gi,dschg_gi_u,dschg_gisf,dschg_gi_q,"
        "dschg_q_b,dschg_gq_b,reach_q,reach_q_b,"
        "dark_frac,ice_clim_f,ice_dyn_f,partial_f,n_good_nod,obs_frac_n,xovr_cal_q,"
        "geoid_hght,geoid_slop,solid_tide,load_tidef,load_tideg,pole_tide,"
        "dry_trop_c,wet_trop_c,iono_c,xovr_cal_c,"
        "n_reach_up,n_reach_dn,rch_id_up,rch_id_dn,"
        "p_wse,p_wse_var,p_width,p_wid_var,p_n_nodes,p_dist_out,"
        "p_length,p_maf,p_dam_id,p_n_ch_max,p_n_ch_mod,p_low_slp,"
        "geometry"
    )
    fields = ",".join([f.strip() for f in all_fields.split(',') if f.strip()])
    
    output_dir = Path(output_file).parent / "reach_chunks_nb"
    output_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"Downloading {len(reach_ids):,} reaches (max_workers={max_workers})...")
    
    chunk_num = 0
    chunk_results = []
    success = 0
    failed = 0
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(
                query_hydrocron, rid, HYDROCRON_URL, START_TIME, END_TIME,
                fields, COLLECTION_NAME, "Reach"
            ): rid
            for rid in reach_ids
        }
        
        for future in as_completed(futures):
            result = future.result()
            if result is not None and len(result) > 0:
                chunk_results.append(result)
                success += 1
                if len(chunk_results) >= chunk_size:
                    chunk_df = pd.concat(chunk_results, ignore_index=True)
                    chunk_df.to_parquet(output_dir / f"chunk_{chunk_num:04d}.parquet", index=False)
                    clear_output(wait=True)
                    print(f"Saved chunk {chunk_num}: {len(chunk_df):,} rows | "
                          f"Progress: {success+failed}/{len(reach_ids)}")
                    del chunk_df, chunk_results
                    chunk_results = []
                    chunk_num += 1
                    gc.collect()
            else:
                failed += 1
    
    # Save remaining
    if chunk_results:
        chunk_df = pd.concat(chunk_results, ignore_index=True)
        chunk_df.to_parquet(output_dir / f"chunk_{chunk_num:04d}.parquet", index=False)
        del chunk_df, chunk_results
        gc.collect()
    
    print(f"\nDownload complete: success={success:,}, failed={failed:,}")
    
    # Merge chunks
    chunks = sorted(output_dir.glob("chunk_*.parquet"))
    all_dfs = []
    for c in chunks:
        df = pd.read_parquet(c)
        df = df[df['wse'] != -999999999999.0]
        all_dfs.append(df)
    
    merged = pd.concat(all_dfs, ignore_index=True)
    merged.to_parquet(output_file, index=False)
    print(f"Saved merged: {output_file} ({len(merged):,} rows)")
    return merged


# ---- Uncomment to run in-notebook download ----
# swot_reach = download_swot_reaches_notebook(study_reach_ids, str(SWOT_REACH_FILE))
print("In-notebook download function defined (uncomment to use).")

---
## Phase 3: USGS Gage Height (Time-Targeted)

### 3.1 Time-Targeted USGS Download

Instead of downloading full 2023-2026 for every gage, only download windows around SWOT observation times.

In [ ]:
# ---- Build per-gage download windows from SWOT observation times ----

# Get SWOT times per reach
if 'time_dt' not in swot_reach.columns:
    swot_reach['time_dt'] = pd.to_datetime(swot_reach['time_str'])

swot_times_by_reach = swot_reach.groupby('reach_id')['time_dt'].apply(list).to_dict()

# Link reach -> gage via id_link
# id_link has columns: reach_id, gage_id
# Filter id_link to study reaches with USGS gages only
id_link_usgs = id_link[
    id_link['reach_id'].isin(study_reach_ids) & 
    id_link['gage_id'].notna()
].copy()
id_link_usgs = id_link_usgs[id_link_usgs['gage_id'].isin(paired_gage_ids)]

print(f"USGS-linked study reaches: {id_link_usgs['reach_id'].nunique():,}")
print(f"USGS gages to download   : {id_link_usgs['gage_id'].nunique():,}")


def build_download_windows(gage_id, reach_ids_for_gage, swot_times_by_reach, pad_days=1):
    """Build merged time windows for a single gage based on SWOT obs times."""
    all_times = []
    for rid in reach_ids_for_gage:
        if rid in swot_times_by_reach:
            all_times.extend(swot_times_by_reach[rid])
    
    if not all_times:
        return []
    
    all_times = sorted(set(all_times))
    pad = timedelta(days=pad_days)
    
    # Create windows
    windows = [(t - pad, t + pad) for t in all_times]
    
    # Merge overlapping windows
    merged = [windows[0]]
    for start, end in windows[1:]:
        if start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    
    return merged


# Build windows per gage
gage_to_reaches = id_link_usgs.groupby('gage_id')['reach_id'].apply(list).to_dict()

gage_windows = {}
for gage_id, reach_ids_for_gage in gage_to_reaches.items():
    windows = build_download_windows(gage_id, reach_ids_for_gage, swot_times_by_reach)
    if windows:
        gage_windows[gage_id] = windows

total_windows = sum(len(w) for w in gage_windows.values())
print(f"\nGages with download windows: {len(gage_windows):,}")
print(f"Total download windows     : {total_windows:,}")

In [ ]:
# ---- Download USGS gage height (instantaneous, parameterCd=00065) ----

def download_usgs_for_gage(gage_id, windows):
    """Download USGS instantaneous gage height for specified time windows."""
    all_records = []
    for w_start, w_end in windows:
        try:
            df = nwis.get_record(
                sites=gage_id,
                service='iv',
                start=w_start.strftime('%Y-%m-%d'),
                end=w_end.strftime('%Y-%m-%d'),
                parameterCd='00065'
            )
            if df is not None and len(df) > 0:
                all_records.append(df)
        except Exception:
            pass
    
    if all_records:
        combined = pd.concat(all_records)
        combined = combined[~combined.index.duplicated(keep='first')]
        combined['gage_id'] = gage_id
        return combined
    return None


# Execute download
print(f"Downloading USGS gage height for {len(gage_windows):,} gages...")
t0 = time.time()

usgs_results = []
success_count = 0
failed_count = 0

for i, (gage_id, windows) in enumerate(gage_windows.items()):
    result = download_usgs_for_gage(gage_id, windows)
    if result is not None and len(result) > 0:
        usgs_results.append(result)
        success_count += 1
    else:
        failed_count += 1
    
    if (i + 1) % 50 == 0 or (i + 1) == len(gage_windows):
        clear_output(wait=True)
        print(f"USGS download: {i+1}/{len(gage_windows)} gages "
              f"(success={success_count}, failed={failed_count}, "
              f"{time.time()-t0:.0f}s)")

print(f"\nUSGS download complete:")
print(f"  Success: {success_count:,} gages")
print(f"  Failed : {failed_count:,} gages")

if usgs_results:
    usgs_raw = pd.concat(usgs_results)
    usgs_raw = usgs_raw.reset_index()
    print(f"  Total USGS records: {len(usgs_raw):,}")
else:
    print("  No USGS data downloaded.")
    usgs_raw = pd.DataFrame()

In [ ]:
# ---- Clean USGS data ----

if len(usgs_raw) > 0:
    # Identify the gage height column (usually '00065' or contains 'gage_ht')
    gh_col = None
    for col in usgs_raw.columns:
        if '00065' in str(col):
            gh_col = col
            break
    
    if gh_col is None:
        # Fallback: try common column names
        for col in usgs_raw.columns:
            if 'gage' in col.lower() and 'height' in col.lower():
                gh_col = col
                break
    
    print(f"Gage height column: {gh_col}")
    print(f"All columns: {usgs_raw.columns.tolist()}")
    
    # Rename and convert units (feet -> meters)
    usgs_clean = usgs_raw[['datetime', 'gage_id', gh_col]].copy()
    usgs_clean = usgs_clean.rename(columns={gh_col: 'gage_height_ft'})
    usgs_clean['datetime'] = pd.to_datetime(usgs_clean['datetime'], utc=True)
    
    n_before_clean = len(usgs_clean)
    usgs_clean = usgs_clean.dropna(subset=['gage_height_ft'])
    n_after_clean = len(usgs_clean)
    print(f"\nDrop NaN gage height: {n_before_clean:,} -> {n_after_clean:,} "
          f"(removed {n_before_clean - n_after_clean:,})")
    
    # Convert feet to meters
    usgs_clean['gage_height_m'] = usgs_clean['gage_height_ft'] * 0.3048
    
    usgs_clean = usgs_clean.sort_values(['gage_id', 'datetime']).reset_index(drop=True)
    print(f"\nUSGS clean data: {len(usgs_clean):,} records, "
          f"{usgs_clean['gage_id'].nunique():,} gages")
else:
    usgs_clean = pd.DataFrame()
    print("No USGS data to clean.")

### 3.2 Temporal Matching (SWOT <-> USGS)

In [ ]:
# ---- Temporal matching via merge_asof ----

if len(usgs_clean) > 0 and len(swot_reach) > 0:
    # Prepare SWOT side: link reach_id -> gage_id
    swot_for_match = swot_reach[['reach_id', 'time_dt', 'wse']].copy()
    swot_for_match = swot_for_match.merge(id_link_usgs, on='reach_id', how='inner')
    swot_for_match['time_dt'] = pd.to_datetime(swot_for_match['time_dt'], utc=True)
    swot_for_match = swot_for_match.sort_values(['gage_id', 'time_dt']).reset_index(drop=True)
    
    # Prepare USGS side
    usgs_for_match = usgs_clean[['gage_id', 'datetime', 'gage_height_m']].copy()
    usgs_for_match = usgs_for_match.sort_values(['gage_id', 'datetime']).reset_index(drop=True)
    
    print(f"SWOT observations for matching: {len(swot_for_match):,}")
    print(f"USGS records for matching     : {len(usgs_for_match):,}")
    
    # merge_asof per gage_id
    matched_list = []
    gage_ids_to_match = swot_for_match['gage_id'].unique()
    
    for gid in gage_ids_to_match:
        swot_g = swot_for_match[swot_for_match['gage_id'] == gid].copy()
        usgs_g = usgs_for_match[usgs_for_match['gage_id'] == gid].copy()
        
        if len(usgs_g) == 0:
            continue
        
        merged = pd.merge_asof(
            swot_g.sort_values('time_dt'),
            usgs_g.sort_values('datetime'),
            left_on='time_dt',
            right_on='datetime',
            by='gage_id',
            tolerance=pd.Timedelta(TEMPORAL_TOLERANCE),
            direction='nearest'
        )
        matched_list.append(merged)
    
    if matched_list:
        matched = pd.concat(matched_list, ignore_index=True)
        
        n_total_swot = len(swot_for_match)
        n_matched = matched['gage_height_m'].notna().sum()
        n_unmatched = n_total_swot - n_matched
        
        print(f"\nMatched: {n_matched:,} out of {n_total_swot:,} SWOT observations "
              f"({n_matched/n_total_swot*100:.1f}%)")
        print(f"Unmatched (no USGS within {TEMPORAL_TOLERANCE}): {n_unmatched:,} "
              f"({n_unmatched/n_total_swot*100:.1f}%)")
        
        # Drop unmatched
        matched = matched.dropna(subset=['gage_height_m'])
        print(f"Final matched dataset: {len(matched):,} rows")
    else:
        matched = pd.DataFrame()
        print("No matches found.")
else:
    matched = pd.DataFrame()
    print("Skipping temporal matching (missing data).")

### 3.3 Relative WSE (Anomaly)

SWOT WSE is geoid-referenced; USGS gage height is datum-referenced.  
We subtract per-reach/per-gage medians to make them comparable as relative (anomaly) values.

In [ ]:
if len(matched) > 0:
    # ---- SWOT relative WSE (subtract per-reach median) ----
    swot_median_per_reach = matched.groupby('reach_id')['wse'].transform('median')
    matched['swot_wse_relative'] = matched['wse'] - swot_median_per_reach
    
    # ---- USGS relative gage height (subtract per-gage median) ----
    usgs_median_per_gage = matched.groupby('gage_id')['gage_height_m'].transform('median')
    matched['usgs_wse_relative'] = matched['gage_height_m'] - usgs_median_per_gage
    
    # ---- WSE error ----
    matched['wse_error'] = matched['swot_wse_relative'] - matched['usgs_wse_relative']
    
    print(f"Relative WSE and error computed for {len(matched):,} observations.")
    print(f"\nWSE error statistics (SWOT_relative - USGS_relative):")
    print(matched['wse_error'].describe())
    print(f"\n  Mean error   : {matched['wse_error'].mean():.4f} m")
    print(f"  Median error : {matched['wse_error'].median():.4f} m")
    print(f"  Std          : {matched['wse_error'].std():.4f} m")
    print(f"  MAE          : {matched['wse_error'].abs().mean():.4f} m")
    print(f"  RMSE         : {np.sqrt((matched['wse_error']**2).mean()):.4f} m")
else:
    print("No matched data available for relative WSE computation.")

In [ ]:
# ---- Merge matched USGS data back into full SWOT reach dataframe ----

if len(matched) > 0:
    # Select columns from matched to merge
    match_cols = ['reach_id', 'time_dt', 'gage_id', 'gage_height_m',
                  'swot_wse_relative', 'usgs_wse_relative', 'wse_error']
    matched_slim = matched[match_cols].copy()
    
    # Merge into swot_reach
    n_before_merge = len(swot_reach)
    df = swot_reach.merge(
        matched_slim,
        on=['reach_id', 'time_dt'],
        how='left'
    )
    n_after_merge = len(df)
    
    n_with_usgs = df['wse_error'].notna().sum()
    n_without_usgs = df['wse_error'].isna().sum()
    
    print(f"Merged USGS into SWOT reach data:")
    print(f"  Rows        : {n_after_merge:,} (was {n_before_merge:,})")
    print(f"  With USGS   : {n_with_usgs:,} ({n_with_usgs/n_after_merge*100:.1f}%)")
    print(f"  Without USGS: {n_without_usgs:,} ({n_without_usgs/n_after_merge*100:.1f}%)")
else:
    df = swot_reach.copy()
    print("No matched data to merge. Using SWOT data only.")

---
## Phase 4: Design Filter + Random Train/Test Split

Per Colin: Quality flags (reach_q, dark_frac, partial_f, etc.) become features, not filters.  
Only remove physically unreliable data:
- Ice-covered observations (ice_clim_f >= 1)
- Cross-track distance outside 10-60 km range
- Obviously corrupted (reach_q == 3)

Then random train/test split (80/20).

In [ ]:
# ---- Phase 4.1: Design limitation filters (unconditional removal) ----
print("Phase 4.1: Design Limitation Filters")

n_before = len(swot_reach)
design_mask = pd.Series(True, index=swot_reach.index)

# [1] Ice climatology
if 'ice_clim_f' in swot_reach.columns:
    f1 = swot_reach['ice_clim_f'] >= 1
    design_mask &= ~f1
    print(f"  [1] Ice climatology (ice_clim_f >= 1): {f1.sum():,} removed")

# [2] Cross-track distance
if 'xtrk_dist' in swot_reach.columns:
    f2 = (swot_reach['xtrk_dist'].abs() < 10000) | (swot_reach['xtrk_dist'].abs() > 60000)
    design_mask &= ~f2
    print(f"  [2] Cross-track distance (<10km or >60km): {f2.sum():,} removed")

# [3] Obviously corrupted (reach_q == 3)
if 'reach_q' in swot_reach.columns:
    f3 = swot_reach['reach_q'] == 3
    design_mask &= ~f3
    print(f"  [3] Obviously corrupted (reach_q == 3): {f3.sum():,} removed")

df_clean = swot_reach[design_mask].copy()
df_removed = swot_reach[~design_mask].copy()
n_after = len(df_clean)

print(f"Design filters: {n_before:,} -> {n_after:,}")
print(f"  Removed: {n_before - n_after:,} ({(n_before - n_after)/n_before*100:.1f}%)")
print(f"  Kept for modeling: {n_after:,} ({n_after/n_before*100:.1f}%)")
print(f"\nNote: Quality flags (dark_frac, partial_f, xovr_cal_q, etc.)")
print(f"      are used as FEATURES, not filters.")

In [ ]:
# ---- Phase 4.2: Random Train/Test Split (80/20) ----
TEST_SIZE = 0.20
RANDOM_STATE = 42

print(f"Phase 4.2: Random Train/Test Split ({(1-TEST_SIZE)*100:.0f}/{TEST_SIZE*100:.0f})")

# Filter to only rows that have USGS matching (wse_error is not NaN)
df_with_usgs = df_clean[df_clean['wse_error'].notna()].copy()
df_without_usgs = df_clean[df_clean['wse_error'].isna()].copy()

print(f"Observations with USGS match: {len(df_with_usgs):,}")
print(f"Observations without USGS:    {len(df_without_usgs):,}")

# Random split
rng = np.random.RandomState(RANDOM_STATE)
n_total = len(df_with_usgs)
indices = rng.permutation(n_total)
n_test = int(n_total * TEST_SIZE)

test_idx = df_with_usgs.index[indices[:n_test]]
train_idx = df_with_usgs.index[indices[n_test:]]

train_df = df_with_usgs.loc[train_idx].copy()
test_df = df_with_usgs.loc[test_idx].copy()

print(f"\nTrain: {len(train_df):,} observations ({len(train_df)/n_total*100:.1f}%)")
print(f"Test:  {len(test_df):,} observations ({len(test_df)/n_total*100:.1f}%)")
print(f"\nTrain reaches: {train_df['reach_id'].nunique():,}")
print(f"Test reaches:  {test_df['reach_id'].nunique():,}")

---
## Phase 5: Feature Engineering

~33 features in 7 groups:
1. SWOT observation geometry (10): xtrk_dist, layovr_val, sig0, dark_frac, area_total, area_detct, obs_frac_n, n_good_nod, loc_offset, node_dist
2. Urban (1): fctimp_mean
3. SWORD prior (8): p_width, p_wse, p_wse_var, p_wid_var, p_n_nodes, p_length, p_dist_out, p_low_slp
4. Geometry (2): p_lat, p_lon
5. Seasonal (4): month_sin, month_cos, doy_sin, doy_cos
6. Reach stats (4): wse_mean, wse_std, width_mean, n_observations
7. Quality flags (7): reach_q, reach_q_b, partial_f, xovr_cal_q, dark_frac, obs_frac_n, wse_prior_diff

Quality flags are FEATURES. The QRF learns to use observation quality  
to separate SWOT radar error from urban geometry effects.

In [ ]:
# ---- Phase 5: Feature Engineering ----
print("Phase 5: Feature Engineering")

from qrf_wse.features import FeatureEngineer, ALL_FEATURES

engineer = FeatureEngineer()

# Prepare train features
X_train, y_train = engineer.prepare_features(train_df)
print(f"\nTrain: {X_train.shape[0]:,} samples x {X_train.shape[1]} features")

# Prepare test features
X_test, y_test = engineer.prepare_features(test_df)
print(f"Test:  {X_test.shape[0]:,} samples x {X_test.shape[1]} features")

# Feature summary
print(f"\nFeature groups:")
print(f"  SWOT observation : {sum(1 for f in ALL_FEATURES if f in ['xtrk_dist','layovr_val','sig0','dark_frac','area_total','area_detct','obs_frac_n','n_good_nod','loc_offset','node_dist'])}")
print(f"  Urban (FctImp)   : 1")
print(f"  SWORD prior      : {sum(1 for f in ALL_FEATURES if f.startswith('p_'))}")
print(f"  Geometry         : 2")
print(f"  Seasonal         : 4")
print(f"  Reach statistics : 4")
print(f"  Quality flags    : {sum(1 for f in ALL_FEATURES if f in ['reach_q','reach_q_b','partial_f','xovr_cal_q','obs_frac_n','wse_prior_diff'])}")
print(f"  TOTAL            : {len(ALL_FEATURES)}")

# Check for NaN
nan_pct = (X_train.isna().sum() / len(X_train) * 100).sort_values(ascending=False)
if nan_pct.max() > 0:
    print(f"\nFeatures with NaN (before imputation):")
    for feat, pct in nan_pct[nan_pct > 0].items():
        print(f"  {feat}: {pct:.1f}%")
else:
    print("\nNo NaN values in features (all imputed).")

---
## Phase 6: QRF Training + k-Fold Cross-Validation

Quantile Regression Forest (Meinshausen 2006):
- 500 trees, min_samples_leaf=10
- Quantiles: 5th (lower bound), 50th (bias correction), 95th (upper bound)
- k-fold CV within training set to verify stability

In [ ]:
# ---- Phase 6.1: Train QRF ----
from sklearn.ensemble import RandomForestRegressor

print("Phase 6.1: QRF Training")

N_ESTIMATORS = 500
MIN_SAMPLES_LEAF = 10
QUANTILES = [0.05, 0.50, 0.95]

# Train the forest
rf = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    max_features=1.0,
    random_state=42,
    n_jobs=-1,
)

print(f"Training QRF: {N_ESTIMATORS} trees, min_leaf={MIN_SAMPLES_LEAF}")
print(f"  {X_train.shape[0]:,} samples x {X_train.shape[1]} features")

t0 = time.time()
rf.fit(X_train.values, y_train.values)
elapsed = time.time() - t0

print(f"  Training time: {elapsed:.1f}s")
print(f"  Mean prediction (sklearn default): {rf.predict(X_train.values[:5])}")

# Predict quantiles via per-tree predictions
def predict_quantiles(model, X, quantiles=[0.05, 0.50, 0.95]):
    """Collect individual-tree predictions and compute empirical quantiles."""
    X_arr = X.values if hasattr(X, 'values') else X
    n_trees = len(model.estimators_)
    n_samples = X_arr.shape[0]
    
    tree_preds = np.empty((n_trees, n_samples))
    for i, tree in enumerate(model.estimators_):
        tree_preds[i] = tree.predict(X_arr)
    
    return {q: np.percentile(tree_preds, q * 100, axis=0) for q in quantiles}

print("\nPredicting quantiles on test set...")
q_test = predict_quantiles(rf, X_test, QUANTILES)
q_train = predict_quantiles(rf, X_train, QUANTILES)

bias_pred_test = q_test[0.50]
bias_pred_train = q_train[0.50]

print(f"  Test median bias prediction: mean={bias_pred_test.mean():.3f}, std={bias_pred_test.std():.3f}")

In [ ]:
# ---- Phase 6.2: k-Fold Cross-Validation ----
from sklearn.model_selection import KFold

N_FOLDS = 5
print(f"\nPhase 6.2: {N_FOLDS}-Fold Cross-Validation")

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
cv_results = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train), 1):
    X_tr = X_train.iloc[tr_idx]
    y_tr = y_train.iloc[tr_idx]
    X_va = X_train.iloc[va_idx]
    y_va = y_train.iloc[va_idx]
    
    fold_rf = RandomForestRegressor(
        n_estimators=N_ESTIMATORS,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        max_features=1.0,
        random_state=42,
        n_jobs=-1,
    )
    fold_rf.fit(X_tr.values, y_tr.values)
    
    # Median prediction via per-tree
    q_va = predict_quantiles(fold_rf, X_va, [0.50])
    y_pred = q_va[0.50]
    
    residual = y_va.values - y_pred
    rmse = np.sqrt(np.mean(residual**2))
    mae = np.mean(np.abs(residual))
    bias = np.mean(residual)
    
    cv_results.append({'fold': fold, 'rmse': rmse, 'mae': mae, 'bias': bias})
    print(f"  Fold {fold}/{N_FOLDS}: RMSE={rmse:.4f}m, MAE={mae:.4f}m, Bias={bias:.4f}m")

cv_df = pd.DataFrame(cv_results)
print(f"\nCV Summary:")
print(f"  RMSE: {cv_df['rmse'].mean():.4f} +/- {cv_df['rmse'].std():.4f} m")
print(f"  MAE:  {cv_df['mae'].mean():.4f} +/- {cv_df['mae'].std():.4f} m")
print(f"  Bias: {cv_df['bias'].mean():.4f} +/- {cv_df['bias'].std():.4f} m")

In [ ]:
# ---- Phase 6.3: Feature Importance ----
print("Phase 6.3: Feature Importance")

importance = rf.feature_importances_
fi_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': importance
}).sort_values('importance', ascending=False).reset_index(drop=True)

print("\nTop 15 features:")
for i, row in fi_df.head(15).iterrows():
    bar = chr(9608) * int(row['importance'] * 200)
    print(f"  {i+1:2d}. {row['feature']:20s} {row['importance']:.4f} {bar}")

In [ ]:
# ---- Phase 6.4: Test Set Evaluation ----
print("Phase 6.4: Test Set Evaluation")

# Before correction
residual_before = y_test.values
rmse_before = np.sqrt(np.mean(residual_before**2))
mae_before = np.mean(np.abs(residual_before))

# After correction (subtract predicted bias)
corrected_error = y_test.values - bias_pred_test
rmse_after = np.sqrt(np.mean(corrected_error**2))
mae_after = np.mean(np.abs(corrected_error))

improvement = (rmse_before - rmse_after) / rmse_before * 100

print(f"\n{'Metric':<20s} {'Before':>12s} {'After':>12s} {'Improvement':>12s}")
print(f"{'RMSE (m)':<20s} {rmse_before:>12.4f} {rmse_after:>12.4f} {improvement:>11.1f}%")
print(f"{'MAE (m)':<20s} {mae_before:>12.4f} {mae_after:>12.4f} {(mae_before-mae_after)/mae_before*100:>11.1f}%")
print(f"{'Bias (m)':<20s} {np.mean(residual_before):>12.4f} {np.mean(corrected_error):>12.4f}")

---
## Phase 7: WSE Correction & Physics Recalculation

Per Colin: "노드 수준에서 보정하고, reach 수준 SP를 재계산해서 얼마나 달라지는지 봐라"

Steps:
1. Apply QRF bias correction to WSE (all data)
2. Width: SWORD prior filter -> clean H-W fit -> corrected width
3. Slope: 1st-order polyfit on corrected WSE vs distance
4. slope2: Gaussian smoothing (sigma=2km) + finite diff
5. d_x_area: Piecewise integration of H-W relationship

In [ ]:
# ---- Phase 7.1: Apply WSE Bias Correction ----
print("Phase 7.1: WSE Bias Correction")

# Apply to ALL data (train + test + no-USGS)
X_all, _ = engineer.prepare_features(df_clean)
q_all = predict_quantiles(rf, X_all, QUANTILES)

df_corrected = df_clean.loc[X_all.index].copy()
df_corrected['wse_corrected'] = df_corrected['wse'] - q_all[0.50]
df_corrected['wse_bias_predicted'] = q_all[0.50]
df_corrected['wse_uncertainty'] = (q_all[0.95] - q_all[0.05]) / 2.0

print(f"Corrected {len(df_corrected):,} observations")
print(f"  Mean bias correction: {q_all[0.50].mean():.3f} m")
print(f"  Mean uncertainty (90% PI): {df_corrected['wse_uncertainty'].mean():.3f} m")

In [ ]:
# ---- Phase 7.2: Width Correction (SWORD-prior filtered H-W fit) ----
from qrf_wse.physics import PhysicsCorrector
physics = PhysicsCorrector()

print("Phase 7.2: Width Correction")

n_reaches = 0
n_corrected = 0

for reach_id, grp in df_corrected.groupby('reach_id'):
    idx = grp.index
    n_reaches += 1
    
    wse_corr = grp['wse_corrected'].values
    w_raw = grp['width'].values if 'width' in grp.columns else None
    w_prior = grp['p_width'].values if 'p_width' in grp.columns else None
    s_raw = grp['slope'].values if 'slope' in grp.columns else np.zeros(len(grp))
    
    if w_raw is not None and w_prior is not None:
        w_corr = physics.correct_width(wse_corr, w_raw, s_raw, w_prior)
        df_corrected.loc[idx, 'width_corrected'] = w_corr
        n_corrected += 1

print(f"Width corrected: {n_corrected:,} / {n_reaches:,} reaches")

In [ ]:
# ---- Phase 7.3: Slope & d_x_area Recomputation ----
print("Phase 7.3: Slope & d_x_area Recomputation")

for reach_id, grp in df_corrected.groupby('reach_id'):
    idx = grp.index
    wse_corr = grp['wse_corrected'].values
    
    # Approximate node distances (use node_dist if available, else estimate)
    if 'node_dist' in grp.columns:
        node_dist = grp['node_dist'].values
    else:
        node_dist = np.arange(len(grp), dtype=float) * 200.0
    
    # Slope (1st-order polyfit)
    slope_corr = physics.compute_slope(wse_corr, node_dist)
    df_corrected.loc[idx, 'slope_corrected'] = slope_corr
    
    # d_x_area (piecewise integration)
    if 'width_corrected' in df_corrected.columns:
        w_corr = df_corrected.loc[idx, 'width_corrected'].values
        d_x_a = physics.compute_d_x_area(wse_corr, w_corr)
        df_corrected.loc[idx, 'd_x_area_corrected'] = d_x_a

print(f"Slope and d_x_area recomputed for {df_corrected['reach_id'].nunique():,} reaches")

# Save corrected output
output_path = "/nas/cee-hydro/floods/vD/qrf_corrected_reaches_vD.parquet"
df_corrected.to_parquet(output_path, index=False)
print(f"\nSaved: {output_path}")

---
## Phase 8: Validation

Compare corrected vs original WSE against USGS ground truth.  
Stratified by FctImp to show urban-specific improvement.

In [ ]:
# ---- Phase 8.1: Overall Validation ----
print("Phase 8.1: Overall Validation (Test Set)")

# Get test-set corrected values
test_corrected = df_corrected.loc[df_corrected.index.isin(test_df.index)].copy()
test_corrected = test_corrected[test_corrected['wse_error'].notna()]

# Before vs After
error_before = test_corrected['wse_error'].values  # SWOT - USGS (relative)
error_after = error_before - test_corrected['wse_bias_predicted'].values

metrics = {
    'Before': {
        'RMSE': np.sqrt(np.mean(error_before**2)),
        'MAE': np.mean(np.abs(error_before)),
        'Bias': np.mean(error_before),
        'Std': np.std(error_before),
    },
    'After': {
        'RMSE': np.sqrt(np.mean(error_after**2)),
        'MAE': np.mean(np.abs(error_after)),
        'Bias': np.mean(error_after),
        'Std': np.std(error_after),
    },
}

print(f"\n{'Metric':<10s} {'Before':>10s} {'After':>10s} {'Change':>10s}")
for metric in ['RMSE', 'MAE', 'Bias', 'Std']:
    before = metrics['Before'][metric]
    after = metrics['After'][metric]
    change = (before - after) / abs(before) * 100 if before != 0 else 0
    print(f"{metric:<10s} {before:>10.4f} {after:>10.4f} {change:>9.1f}%")

print(f"\nN observations: {len(test_corrected):,}")
print(f"N reaches:      {test_corrected['reach_id'].nunique():,}")

In [ ]:
# ---- Phase 8.2: FctImp-Stratified Validation ----
print("Phase 8.2: FctImp-Stratified Validation")

# Merge FctImp into test_corrected if not already present
if 'fctimp_mean' not in test_corrected.columns:
    fctimp_lookup = sword[['reach_id', 'fctimp_mean']].drop_duplicates()
    test_corrected = test_corrected.merge(fctimp_lookup, on='reach_id', how='left')

# Create FctImp bins
bins = [0, 5, 15, 30, 50, 100]
labels = ['0-5%', '5-15%', '15-30%', '30-50%', '50-100%']
test_corrected['fctimp_bin'] = pd.cut(
    test_corrected['fctimp_mean'], bins=bins, labels=labels, include_lowest=True
)

print(f"\n{'FctImp Bin':<12s} {'N':>6s} {'RMSE_before':>12s} {'RMSE_after':>12s} {'Improve%':>10s}")

for bin_label in labels:
    subset = test_corrected[test_corrected['fctimp_bin'] == bin_label]
    if len(subset) < 5:
        continue
    
    err_b = subset['wse_error'].values
    err_a = err_b - subset['wse_bias_predicted'].values
    
    rmse_b = np.sqrt(np.mean(err_b**2))
    rmse_a = np.sqrt(np.mean(err_a**2))
    improve = (rmse_b - rmse_a) / rmse_b * 100
    
    print(f"{bin_label:<12s} {len(subset):>6,} {rmse_b:>12.4f} {rmse_a:>12.4f} {improve:>9.1f}%")

In [ ]:
# ---- Phase 8.3: Prediction Interval Calibration ----
print("Phase 8.3: Prediction Interval Calibration (90% PI)")

# Check if actual errors fall within the predicted 5th-95th interval
test_q = predict_quantiles(rf, X_test, QUANTILES)
pi_lower = test_q[0.05]
pi_upper = test_q[0.95]

actual_error = y_test.values
within_pi = (actual_error >= pi_lower) & (actual_error <= pi_upper)
coverage = within_pi.mean() * 100

print(f"\n90% Prediction Interval:")
print(f"  Expected coverage: 90.0%")
print(f"  Actual coverage:   {coverage:.1f}%")
print(f"  {'WELL CALIBRATED' if 85 <= coverage <= 95 else 'NEEDS ADJUSTMENT'}")
print(f"\n  Mean PI width: {(pi_upper - pi_lower).mean():.3f} m")

In [ ]:
print("Notebook complete.")
